# 01 - Collecte des donnees (API France Travail)

**Projet ST2MLE - Analyse des offres d'emploi en France**

Ce notebook collecte les offres d'emploi via l'API officielle **France Travail - Offres d'emploi v2** (source 100% francaise, conforme a la contrainte du sujet).

Deux etapes :
1. Obtenir un **jeton OAuth2** (`client_credentials`).
2. Interroger l'endpoint `/offres/search` avec **pagination** (parametre `range`, max 150 offres/requete, ~3150 max par jeu de filtres).

Pour depasser largement le **minimum de 1 000 lignes** impose, on **boucle departement par departement** et on **deduplique** par identifiant d'offre.

> **Prerequis :** un fichier `.env` (a la racine du projet) avec `FT_CLIENT_ID` et `FT_CLIENT_SECRET`, obtenus sur https://francetravail.io (Mes applications -> abonnement a *Offres d'emploi v2*).

In [ ]:
import os
import time
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv('../.env')
CLIENT_ID = os.getenv('FT_CLIENT_ID')
CLIENT_SECRET = os.getenv('FT_CLIENT_SECRET')
assert CLIENT_ID and CLIENT_SECRET, 'Renseigne FT_CLIENT_ID / FT_CLIENT_SECRET dans .env'

## 1. Authentification OAuth2

On echange l'`id`/`secret` contre un jeton temporaire (~25 min). Le scope `api_offresdemploiv2 o2dsoffre` donne acces a l'API des offres.

In [ ]:
def get_token(client_id, client_secret):
    resp = requests.post(
        'https://entreprise.francetravail.fr/connexion/oauth2/access_token',
        params={'realm': '/partenaire'},
        data={
            'grant_type': 'client_credentials',
            'client_id': client_id,
            'client_secret': client_secret,
            'scope': 'api_offresdemploiv2 o2dsoffre',
        },
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()['access_token']

token = get_token(CLIENT_ID, CLIENT_SECRET)
print('Jeton obtenu :', token[:25], '...')

## 2. Recherche paginee

L'en-tete HTTP `Content-Range` (`offres start-end/total`) indique le nombre total d'offres disponibles pour un filtre donne, ce qui permet de savoir jusqu'ou paginer.

In [ ]:
SEARCH_URL = 'https://api.francetravail.io/partenaire/offresdemploi/v2/offres/search'
PAGE_SIZE = 150        # max autorise par requete
MAX_OFFSET = 3149      # plafond de pagination par jeu de filtres

def search_page(token, params, start, count):
    query = dict(params)
    query['range'] = f'{start}-{start + count - 1}'
    resp = requests.get(
        SEARCH_URL,
        headers={'Authorization': f'Bearer {token}', 'Accept': 'application/json'},
        params=query, timeout=30,
    )
    if resp.status_code == 204:      # aucune offre pour ce filtre
        return [], 0
    resp.raise_for_status()
    offres = resp.json().get('resultats', [])
    total = 0
    cr = resp.headers.get('Content-Range', '')
    if '/' in cr:
        try:
            total = int(cr.split('/')[-1])
        except ValueError:
            total = len(offres)
    return offres, total

def search_all(token, params, pause=0.3):
    """Pagine entierement un jeu de filtres (dans la limite de l'API)."""
    collected, (first, total) = [], search_page(token, params, 0, PAGE_SIZE)
    collected.extend(first)
    limit = min(total, MAX_OFFSET + 1)
    start = PAGE_SIZE
    while start < limit:
        page, _ = search_page(token, params, start, min(PAGE_SIZE, limit - start))
        if not page:
            break
        collected.extend(page)
        start += PAGE_SIZE
        time.sleep(pause)   # respecte la limite de debit de l'API
    return collected

# Test rapide sur Paris
offres_test, total_test = search_page(token, {'departement': '75'}, 0, 3)
print('Offres disponibles a Paris :', total_test)
print('Exemple :', offres_test[0]['intitule'] if offres_test else 'aucune')

## 3. Extraction des champs utiles

Chaque offre est un JSON imbrique. On l'aplatit en gardant :
- les **cibles textuelles** : metier (code/libelle ROME), secteur d'activite (NAF), type de contrat ;
- le **texte** pour le NLP : description, competences ;
- les **variables numeriques** pour predire le salaire : salaire, experience, duree, qualification, localisation.

In [ ]:
def extraire_champs(o):
    lieu = o.get('lieuTravail', {}) or {}
    sal = o.get('salaire', {}) or {}
    ent = o.get('entreprise', {}) or {}
    comp = o.get('competences', []) or []
    return {
        'id': o.get('id'),
        'intitule': o.get('intitule'),
        'rome_code': o.get('romeCode'),
        'rome_libelle': o.get('romeLibelle'),
        'appellation_libelle': o.get('appellationlibelle'),
        'secteur_activite': o.get('secteurActivite'),
        'secteur_activite_libelle': o.get('secteurActiviteLibelle'),
        'type_contrat': o.get('typeContrat'),
        'type_contrat_libelle': o.get('typeContratLibelle'),
        'nature_contrat': o.get('natureContrat'),
        'description': o.get('description'),
        'competences': ' | '.join(c.get('libelle', '') for c in comp if c.get('libelle')),
        'salaire_libelle': sal.get('libelle'),
        'salaire_complement1': sal.get('complement1'),
        'salaire_complement2': sal.get('complement2'),
        'experience_exige': o.get('experienceExige'),
        'experience_libelle': o.get('experienceLibelle'),
        'duree_travail_libelle': o.get('dureeTravailLibelle'),
        'duree_travail_converti': o.get('dureeTravailLibelleConverti'),
        'alternance': o.get('alternance'),
        'qualification_libelle': o.get('qualificationLibelle'),
        'lieu_libelle': lieu.get('libelle'),
        'code_postal': lieu.get('codePostal'),
        'commune': lieu.get('commune'),
        'latitude': lieu.get('latitude'),
        'longitude': lieu.get('longitude'),
        'entreprise_nom': ent.get('nom'),
        'date_creation': o.get('dateCreation'),
        'date_actualisation': o.get('dateActualisation'),
    }

## 4. Collecte complete

Boucle sur tous les departements (metropole + Corse + DOM) et deduplication par `id`.
Compter quelques minutes (pagination + pauses anti rate-limit).

In [ ]:
DEPARTEMENTS = (
    [f'{i:02d}' for i in range(1, 20)] + ['2A', '2B']
    + [f'{i:02d}' for i in range(21, 96)]
    + ['971', '972', '973', '974', '976']
)

token = get_token(CLIENT_ID, CLIENT_SECRET)
vues = {}
for i, dep in enumerate(DEPARTEMENTS, 1):
    offres = search_all(token, {'departement': dep})
    for o in offres:
        if o.get('id'):
            vues[o['id']] = extraire_champs(o)
    print(f'[{i:>3}/{len(DEPARTEMENTS)}] dep {dep:<4} +{len(offres):<4} -> total unique : {len(vues)}')

df = pd.DataFrame(list(vues.values()))
print('\nTotal offres uniques :', len(df))
df.head()

## 5. Apercu des cibles et sauvegarde

In [ ]:
for col in ['rome_libelle', 'secteur_activite_libelle', 'type_contrat_libelle']:
    print('---', col, '---')
    print(df[col].value_counts().head(8), '\n')

In [ ]:
Path('../data/raw').mkdir(parents=True, exist_ok=True)
df.to_csv('../data/raw/offres_brut.csv', index=False, encoding='utf-8')
df.to_parquet('../data/raw/offres_brut.parquet', index=False)
print(f'{len(df):,} offres sauvegardees dans data/raw/')